# Adaptive Encoder Analysis

Use this notebook after running `scripts/adaptive_encoder_pipeline.py`. It summarizes the adaptive output-dimension sweep, the sparse reduced encoder, and PySR on encoded invariants `J1...Jn`.

In [ ]:
from pathlib import Path
import json
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

MPLCONFIGDIR = PROJECT_ROOT / ".matplotlib-cache"
MPLCONFIGDIR.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIGDIR))

CONFIG_PATH = PROJECT_ROOT / "configs" / "adaptive_encoder_rotated_hill.toml"
STAGE1_DIR = PROJECT_ROOT / "results" / "adaptive_rotatedhill"
SPARSE_RUN_DIR = PROJECT_ROOT / "results" / "adaptive_rotatedhill_sparse"
STAGE1_SUMMARY = STAGE1_DIR / "adaptive_stage1_summary.json"
STAGE2_SUMMARY = SPARSE_RUN_DIR / "adaptive_stage2_sparsify.json"
SPARSE_CHECKPOINT = SPARSE_RUN_DIR / "checkpoint_best.pt"
SYMBOLIC_DIR = SPARSE_RUN_DIR / "symbolic_encoded"

print("project:", PROJECT_ROOT)
print("config:", CONFIG_PATH)
print("stage1 summary:", STAGE1_SUMMARY)
print("stage2 summary:", STAGE2_SUMMARY)
print("symbolic dir:", SYMBOLIC_DIR)

In [ ]:
import numpy as np
import torch
from pprint import pprint

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

import matplotlib.pyplot as plt

from invariant_generator.config import load_config
from invariant_generator.data import prepare_training_data
from invariant_generator.evaluation import evaluate_model, predict_numpy
from invariant_generator.formulas import encoder_formula_report
from invariant_generator.model import InvariantYieldModel
from invariant_generator.utils import resolve_device

config = load_config(CONFIG_PATH)
device = resolve_device(config.train.device)

## Stage 1: Adaptive n Sweep

In [ ]:
with STAGE1_SUMMARY.open("r", encoding="utf-8") as f:
    stage1 = json.load(f)

runs = stage1.get("runs", [])
metric = stage1.get("metric", "rmse")
selected_n = stage1.get("selected_n")
print("search direction:", stage1.get("search_direction", "forward"))
print("metric:", metric)
print("threshold:", stage1.get("threshold"))
print("max loss delta:", stage1.get("max_loss_delta"))
print("max relative loss delta:", stage1.get("max_relative_loss_delta"))
print("patience:", stage1.get("patience"))
print("reference n:", stage1.get("reference_n"))
print("selected n:", selected_n)
print("selected checkpoint:", stage1.get("selected_checkpoint"))

n_values = np.array([row["n"] for row in runs], dtype=int)
train_values = np.array([row["train_metrics"][metric] for row in runs], dtype=float)
test_values = np.array([row["test_metrics"][metric] for row in runs], dtype=float)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(n_values, train_values, marker="o", label=f"train {metric}")
ax.plot(n_values, test_values, marker="o", label=f"test {metric}")
if selected_n is not None:
    ax.axvline(selected_n, color="black", linestyle="--", linewidth=1, label="selected n")
ax.set_xlabel("encoder output dimension n")
ax.set_ylabel(metric)
ax.set_title("Adaptive encoder sweep")
ax.legend()
fig.tight_layout()
plt.show()

for row in runs:
    print(f"n={row['n']:02d} selected={row['selected']} train_{metric}={row['train_metrics'][metric]:.6g} test_{metric}={row['test_metrics'][metric]:.6g}")

## Stage 2: Sparse Encoder

In [ ]:
with STAGE2_SUMMARY.open("r", encoding="utf-8") as f:
    stage2 = json.load(f)

print("method:", stage2.get("method"))
print("threshold:", stage2.get("threshold"))
print("source checkpoint:", stage2.get("source_checkpoint"))
print("sparse checkpoint:", stage2.get("checkpoint"))
print("train metrics:")
pprint(stage2.get("train_metrics"))
print("test metrics:")
pprint(stage2.get("test_metrics"))

source_S = np.asarray(stage2["source_S"], dtype=float)
dense_S = np.asarray(stage2["trained_dense_S"], dtype=float)
final_S = np.asarray(stage2["final_sparse_S"], dtype=float)
mask = np.asarray(stage2["mask"], dtype=int)
active_counts = np.asarray(stage2["active_counts"], dtype=int)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, values, title in zip(axes, [source_S, dense_S, final_S], ["Stage 1 S", "S after sparsity training", "Final sparse S"]):
    vmax = max(float(np.max(np.abs(values))), 1e-12)
    image = ax.imshow(values, aspect="auto", cmap="coolwarm", vmin=-vmax, vmax=vmax)
    ax.set_title(title)
    ax.set_xlabel("input invariant")
    ax.set_ylabel("encoded invariant")
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(np.arange(1, len(active_counts) + 1), active_counts)
ax.set_xlabel("encoded invariant J_i")
ax.set_ylabel("active terms")
ax.set_title("Sparse terms per encoded invariant")
plt.show()

In [ ]:
formulas = stage2.get("formulas", {}).get("formulas", [])
print("Dense-style final formulas in normalized coordinates:")
for item in formulas:
    print(item["normalized_formula"])

print("\nSparse final formulas in raw invariant coordinates:")
for item in formulas:
    print(item["raw_formula"])
    print("  active:", ", ".join(item["active_terms"]) or "none")

## Prediction and Structure Diagnostics

In [ ]:
sparse_config = load_config(CONFIG_PATH)
sparse_config.encoder.output_dim = final_S.shape[0]
sparse_config.train.run_id = sparse_config.sparsification.run_id
data = prepare_training_data(sparse_config)
model = InvariantYieldModel.from_config(sparse_config).to(device)
checkpoint = torch.load(SPARSE_CHECKPOINT, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"], strict=False)

metrics = evaluate_model(model, data.X_test, data.y_test, device=device, batch_size=8192)
pprint(metrics)
y_pred = predict_numpy(model, data.X_test, device=device, batch_size=8192)
y_true = data.y_test
error = y_pred - y_true

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(y_true, y_pred, s=18, alpha=0.75)
low = min(float(y_true.min()), float(y_pred.min()))
high = max(float(y_true.max()), float(y_pred.max()))
axes[0].plot([low, high], [low, high], color="black", linewidth=1)
axes[0].set_title("Test predictions vs targets")
axes[0].set_xlabel("target")
axes[0].set_ylabel("prediction")
axes[1].hist(error, bins=30)
axes[1].set_title("Test prediction error")
axes[1].set_xlabel("prediction - target")
axes[1].set_ylabel("count")
fig.tight_layout()
plt.show()

## Stage 3: PySR on Encoded Invariants

In [ ]:
metrics_path = SYMBOLIC_DIR / "metrics.json"
formulas_path = SYMBOLIC_DIR / "encoded_invariant_formulas.json"
best_equation_path = SYMBOLIC_DIR / "best_equation.txt"

if metrics_path.exists():
    with metrics_path.open("r", encoding="utf-8") as f:
        symbolic_metrics = json.load(f)
    print("PySR metrics:")
    pprint(symbolic_metrics)
else:
    print("No PySR metrics found yet:", metrics_path)

if best_equation_path.exists():
    print("\nBest equation:")
    print(best_equation_path.read_text(encoding="utf-8"))

if formulas_path.exists():
    with formulas_path.open("r", encoding="utf-8") as f:
        symbolic_formulas = json.load(f)
    print("Encoded invariant formulas used by PySR:")
    for item in symbolic_formulas.get("formulas", []):
        print(item["raw_formula"])